In [ ]:
!pip install pandas==2.2.3 -q


In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login, HfApi, list_repo_files
import pandas as pd

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("Successfully logged in!")

In [ ]:
repo_id = "ARTPARK-IISc/Vaani"
api = HfApi()
info = api.dataset_info(
    repo_id=repo_id,
    token=HF_TOKEN
)
print("Dataset Name:", info.id)
print("Author:", info.author)
print("Private:", info.private)
print("Gated:", info.gated)

In [ ]:
files = list_repo_files(
    repo_id=repo_id,
    repo_type="dataset",
    token=HF_TOKEN
)
print("Total number of files:", len(files))

In [ ]:
for i, file in enumerate(files[:10]):
    print(i, ":", file)

In [ ]:
top_folders = set()
for file in files:
    parts = file.split("/")
    if len(parts) > 1:
        top_folders.add(parts[0])
    else:
        top_folders.add("ROOT FILES")

print("Top-level structure:\n")
for folder in sorted(top_folders):
    print(folder)

In [ ]:
from collections import Counter
folder_count = Counter()
for file in files:
    parts = file.split("/")

    if len(parts) > 1:
        folder_count[parts[0]] += 1
    else:
        folder_count["ROOT FILES"] += 1
print("File distribution:\n")
for folder, count in folder_count.items():
    print(folder, ":", count, "files")

In [ ]:
audio_files = [
    file for file in files
    if file.startswith("audio/")
]
print("Total audio-related files:", len(audio_files))
print("\nFirst 50 audio paths:\n")
for file in audio_files[:10]:
    print(file)

In [ ]:
languages = set()
for file in audio_files:
    parts = file.split("/")

    if len(parts) >= 2:
        languages.add(parts[1])
print("Possible languages/categories found:", len(languages))
for language in sorted(languages):
    print(language)

In [ ]:
language_count = Counter()
for file in audio_files:
    parts = file.split("/")

    if len(parts) >= 2:
        language = parts[1]
        language_count[language] += 1

print("Audio file count by language:\n")
for language, count in language_count.most_common():
    print(f"{language}: {count}")

In [ ]:
marathi_files = [
    file for file in files
    if "marathi" in file.lower()
]

print("Marathi-related files:", len(marathi_files))

for file in marathi_files[:10]:
    print(file)

In [ ]:
hindi_files = [
    file for file in files
    if "hindi" in file.lower()
]
print("Hindi-related files:", len(hindi_files))
for file in hindi_files[:10]:
    print(file)

In [ ]:
csv_files = [
    file for file in files
    if file.endswith(".csv")
]

print("CSV files:", len(csv_files))

for file in csv_files[:30]:
    print(file)

In [ ]:
json_files = [
    file for file in files
    if file.endswith(".json")
]

print("JSON files:", len(json_files))

for file in json_files[:30]:
    print(file)

In [ ]:
parquet_files = [
    file for file in files
    if file.endswith(".parquet")
]

print("Parquet files:", len(parquet_files))

for file in parquet_files[:10]:
    print(file)

In [ ]:
summary = {
    "Total Files": len(files),
    "Audio Files": len(audio_files),
    "Marathi Related Files": len(marathi_files),
    "Hindi Related Files": len(hindi_files),
    "CSV Files": len(csv_files),
    "JSON Files": len(json_files),
    "Parquet Files": len(parquet_files)
}

df_summary = pd.DataFrame(
    list(summary.items()),
    columns=["Category", "Count"]
)

df_summary

In [ ]:
sample_file = audio_files[0]

print("Selected Vaani audio file:")
print(sample_file)

In [ ]:
from huggingface_hub import hf_hub_download

audio_path = hf_hub_download(
    repo_id="ARTPARK-IISc/Vaani",
    repo_type="dataset",
    filename=sample_file,
    token=HF_TOKEN
)

print("Downloaded audio path:")
print(audio_path)

In [ ]:
from IPython.display import Audio, display

display(Audio(audio_path))

In [ ]:
!pip install -q datasets pyarrow soundfile

In [ ]:
import pandas as pd
df = pd.read_parquet(audio_path)
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
df.head()

In [ ]:
print(df.dtypes)

In [ ]:
sample = df.iloc[0]

for column in df.columns:
    print("\nCOLUMN:", column)
    print("VALUE:", sample[column])

In [ ]:
audio_data = df.iloc[0]["audio"]

print(type(audio_data))
print(audio_data.keys())

In [ ]:
audio_bytes = audio_data["bytes"]

with open("/content/vaani_sample.wav", "wb") as f:
    f.write(audio_bytes)

print("Audio extracted successfully!")

In [ ]:
from IPython.display import Audio, display

display(Audio("/content/vaani_sample.wav"))

In [ ]:
!pip install -q openai-whisper
!apt-get update -qq
!apt-get install -y ffmpeg -qq


In [ ]:
import whisper
model = whisper.load_model("base")
print("Whisper loaded successfully!")

In [ ]:
result = model.transcribe("/content/vaani_sample.wav")

print("=" * 50)
print("VAANI SPEECH TO TEXT RESULT")
print("=" * 50)

print("\nDetected Language:")
print(result["language"])

print("\nTranscription:")
print(result["text"])

In [ ]:
audio_bytes = audio_data["bytes"]

with open("/content/vaani_sample.wav", "wb") as f:
    f.write(audio_bytes)

In [ ]:
!ffprobe -v error -show_entries stream=codec_name,sample_rate,channels,duration -of default=noprint_wrappers=1 "/content/vaani_sample.wav"

In [ ]:
!file "/content/vaani_sample.wav"

In [ ]:
from IPython.display import Audio, display

display(Audio("/content/vaani_sample.wav"))

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt

audio, sr = librosa.load("/content/vaani_sample.wav")

plt.figure(figsize=(12, 4))
librosa.display.waveshow(audio, sr=sr)
plt.title("Vaani Audio Waveform")
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
result = model.transcribe(
    "/content/vaani_sample.wav",
    language="mr",
    task="transcribe",
    fp16=False
)

print("Detected Language:", result["language"])
print("Transcription:", result["text"])

In [ ]:
hindi_files = [
    file for file in files
    if "hindi" in file.lower()
]

print("Hindi-related files found:", len(hindi_files))

for i, file in enumerate(hindi_files[:10]):
    print(i, file)

In [ ]:
hindi_parquet_files = [
    file for file in hindi_files
    if file.endswith(".parquet")
]

print("Hindi Parquet files:", len(hindi_parquet_files))

for i, file in enumerate(hindi_parquet_files):
    print(i, file)

In [ ]:
hindi_file = hindi_parquet_files[0]

print("Selected Hindi file:")
print(hindi_file)

In [ ]:
from huggingface_hub import hf_hub_download

hindi_parquet_path = hf_hub_download(
    repo_id="ARTPARK-IISc/Vaani",
    repo_type="dataset",
    filename=hindi_file,
    token=HF_TOKEN
)

print("Downloaded successfully!")
print(hindi_parquet_path)

In [ ]:
!pip install -q datasets pyarrow soundfile

In [ ]:
from datasets import Dataset
hindi_dataset = Dataset.from_parquet(hindi_parquet_path)
print(hindi_dataset)

In [ ]:
audio_sample = hindi_dataset[0]["audio"]
print(audio_sample)

In [ ]:
audio_bytes = audio_sample["bytes"]
with open("/content/vaani_hindi_original", "wb") as f:
    f.write(audio_bytes)

print("Hindi audio extracted!")